# 05 — Feature Engineering (Kỹ thuật Đặc trưng)
**Dự án: HitRadar Pro | Phân hệ: EPIC 2 — Tiền xử lý Dữ liệu (Data Preprocessing)**

---

## 1. MỤC TIÊU VÀ PHƯƠNG PHÁP LUẬN
Giai đoạn Feature Engineering tập trung vào việc chuyển đổi dữ liệu thô từ cơ sở dữ liệu thành các đặc trưng đầu vào (input features) tối ưu cho các mô hình học máy. Dựa trên kết quả Phân tích Dữ liệu Khám phá (EDA - Notebook 04), quy trình này sẽ giải quyết ba vấn đề cốt lõi của tập dữ liệu:

1. **Xử lý Dữ liệu Khuyết (Missing Value Imputation):** Tập dữ liệu có các giá trị bị thiếu ở hai đặc trưng `tempo` và `time_signature`. Phương pháp nội suy sẽ được lựa chọn dựa trên phân phối thống kê của từng biến (Trung vị cho biến liên tục, Yếu vị cho biến rời rạc).
2. **Chuẩn hóa Phân phối (Logarithmic Transformation):** Phân tích hình thái cho thấy các đặc trưng `speechiness` và `instrumentalness` có độ lệch phải (right-skewness) rất cao. Việc áp dụng biến đổi logarit $\log(x+1)$ là cần thiết để giảm thiểu phương sai và hỗ trợ sự hội tụ của thuật toán tối ưu (Gradient Descent).
3. **Đồng bộ hóa Thang đo (Feature Scaling):** Các biến số học trong tập dữ liệu không cùng đơn vị đo lường (ví dụ: `release_year` mang giá trị nghìn, trong khi `danceability` nằm trong khoảng [0, 1]). Áp dụng Min-Max Scaling để đưa toàn bộ ma trận đặc trưng về không gian véc-tơ chuẩn [0, 1].


In [ ]:
import os
import warnings
import psycopg2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler

# --- Thiết lập tham số môi trường ---
warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 120
sns.set_theme(style="whitegrid")
pd.set_option('display.max_columns', 30)

# --- Kết nối Cơ sở Dữ liệu ---
password = os.environ.get("PGPASSWORD")
if not password:
    raise ValueError("Lỗi: Biến môi trường PGPASSWORD chưa được cấu hình.")

try:
    conn = psycopg2.connect(host='localhost', port=5432, user='postgres', password=password, dbname='hitradar')
    print('Trạng thái: Kết nối CSDL PostgreSQL thành công.')
except Exception as e:
    print(f"Trạng thái: Kết nối CSDL thất bại. Chi tiết lỗi: {e}")


### Giải thích, Nhận xét & Đánh giá Chuyên sâu: Kiến trúc Kết nối CSDL

1. GIẢI THÍCH:
Sử dụng thư viện `psycopg2` để thiết lập một kết nối trực tiếp đến Data Warehouse (PostgreSQL) nhằm truy xuất View `analytics.vw_ml_training_dataset`. Dữ liệu truy vấn là dữ liệu đã qua quá trình làm sạch nghiêm ngặt ở Pha 1 (EPIC 1). Thông tin đăng nhập được trích xuất an toàn từ biến môi trường `PGPASSWORD` thông qua module `os`.

2. NHẬN XÉT:
Đây là một mô hình thiết kế Lỏng lẻo (Decoupled Architecture) mang tính chuẩn mực trong Data Engineering. Việc không hard-code (gán cứng) mật khẩu trực tiếp vào mã nguồn mà phụ thuộc vào biến môi trường giúp tăng tính bảo mật (Security) và khả năng triển khai liên tục (CI/CD Deployability) trên các nền tảng đám mây (Cloud) như AWS hoặc GCP mà không cần sửa code.

3. ĐÁNH GIÁ (MEDIUM IMPACT):
Bước thiết lập kết nối này quyết định tính liền mạch của Pipeline Học máy (End-to-end ML Pipeline). Khả năng xử lý tự động lỗi (Exception Handling) với cấu trúc `try-except` đảm bảo tiến trình sẽ cảnh báo ngay lập tức nếu có sự cố về mạng, thay vì để thuật toán sụp đổ giữa chừng gây lãng phí tài nguyên tính toán (Compute Resources).

## 2. TRÍCH XUẤT VÀ KIỂM TRA DỮ LIỆU
Thực hiện truy vấn SQL để tải 13 cột cần thiết phục vụ cho quá trình tiền xử lý và huấn luyện mô hình.

In [ ]:
query = """
    SELECT track_id, target_popularity, duration_min, release_year, 
           danceability, energy, loudness, acousticness, instrumentalness, 
           liveness, valence, tempo, time_signature
    FROM analytics.vw_ml_training_dataset
"""
df = pd.read_sql(query, conn)

print(f"Kích thước DataFrame: {df.shape[0]} dòng, {df.shape[1]} cột")
display(df.head())
display(df.describe().T)


### Giải thích, Nhận xét & Đánh giá Chuyên sâu: Thống kê Mô tả & Phân phối

1. GIẢI THÍCH:
Lệnh `df.describe().T` sinh ra một ma trận tóm tắt thống kê đa chiều cho tất cả các biến định lượng (Quantitative Variables). Ma trận này tính toán Số lượng (Count), Trung bình (Mean), Độ lệch chuẩn (Std), Giá trị nhỏ nhất (Min), lớn nhất (Max) và các tứ phân vị (Quartiles). 

2. NHẬN XÉT:
Ma trận thống kê bộc lộ ba điểm nghẽn dữ liệu cực kỳ nghiêm trọng: 
Thứ nhất, tồn tại sự chênh lệch biên độ (Scale Discrepancy) khổng lồ giữa biến `release_year` (nghìn đơn vị) và `acousticness` (thập phân). 
Thứ hai, biến `tempo` và `time_signature` có hiện tượng Khuyết dữ liệu (Missing Values) do tổng `count` thấp hơn 586K. 
Thứ ba, biến `instrumentalness` có Trung vị (50%) bằng 0, cho thấy phân phối lệch phải (Right-Skewed) nặng nề.

3. ĐÁNH GIÁ (HIGH IMPACT):
Sự chênh lệch thang đo và dữ liệu khuyết là "kẻ thù" của các mô hình học máy. Nếu đưa trực tiếp ma trận thô này vào huấn luyện, thuật toán Tối ưu hóa (Gradient Descent) sẽ mất phương hướng, và các mô hình dựa trên khoảng cách (Distance-based) như K-NN hay SVM sẽ bị chi phối hoàn toàn bởi thuộc tính có biên độ lớn như `release_year`. Việc chuẩn hóa (Scaling) và Nội suy (Imputation) là yêu cầu bắt buộc (Mandatory) ở các bước tiếp theo.

## 3. XỬ LÝ DỮ LIỆU KHUYẾT THIẾU (IMPUTATION)
Quá trình nội suy (imputation) được thực hiện dựa trên đặc điểm định lượng của từng biến nhằm duy trì tính chất thống kê tổng thể:
- **`tempo` (Biến liên tục):** Nhịp độ bài hát có thể chứa các giá trị ngoại lai (outliers). Do đó, sử dụng Trung vị (Median) thay vì Số trung bình (Mean) để điền khuyết giúp ước lượng kháng nhiễu (robust estimation) tốt hơn.
- **`time_signature` (Biến rời rạc):** Nhịp phách là biến phân loại. Việc dùng trung bình sẽ tạo ra các giá trị vô nghĩa (ví dụ: 3.5). Giải pháp chuẩn xác về mặt toán học là sử dụng Yếu vị (Mode) - giá trị có tần suất xuất hiện cao nhất.

In [ ]:
# Kiểm tra số lượng giá trị Null
null_counts = df.isnull().sum()
print("Số lượng giá trị Null trước xử lý:")
display(null_counts[null_counts > 0])

# Xử lý nội suy
tempo_median = df['tempo'].median()
df['tempo'] = df['tempo'].fillna(tempo_median)

time_signature_mode = df['time_signature'].mode()[0]
df['time_signature'] = df['time_signature'].fillna(time_signature_mode)

print(f"\nKết quả nội suy:")
print(f"- Thay thế Null trong 'tempo' bằng Trung vị: {tempo_median:.2f}")
print(f"- Thay thế Null trong 'time_signature' bằng Yếu vị: {time_signature_mode}")
print(f"Tổng số Null hiện tại: {df.isnull().sum().sum()}")


### Giải thích, Nhận xét & Đánh giá Chuyên sâu: Xử lý Dữ liệu Khuyết (Missing Value Imputation)

1. GIẢI THÍCH:
Thực hiện kỹ thuật Nội suy (Imputation) để lấp đầy các lỗ hổng dữ liệu (Null/NaN) trong ma trận. Cụ thể, thuật toán sử dụng Trung vị (Median) bằng 121 BPM để điền cho biến liên tục `tempo`, và sử dụng Yếu vị (Mode) bằng 4 để điền cho biến phân loại `time_signature`.

2. NHẬN XÉT:
Việc sử dụng Trung vị (Median) thay cho Số Trung bình (Mean) cho biến `tempo` là một quyết định sắc sảo về mặt toán học. Trung vị có khả năng kháng nhiễu (Robustness against Outliers) tốt hơn rất nhiều, giúp dữ liệu không bị kéo lệch bởi những bài hát có nhịp điệu quá nhanh hoặc quá chậm. Tương tự, Yếu vị 4 cho nhịp phách phản ánh chính xác cấu trúc nhịp 4/4 phổ biến nhất trong lịch sử âm nhạc đại chúng.

3. ĐÁNH GIÁ (CRITICAL IMPACT):
Kỹ thuật nội suy thông minh này giúp làm đặc (Densely Populate) không gian dữ liệu mà không làm méo mó (Distort) sự phân bố gốc. Nếu chọn cách xóa bỏ (Drop) các hàng bị Null, chúng ta sẽ làm mất đi một khối lượng lớn thông tin giá trị từ các đặc trưng khác. Tập dữ liệu nay đã hoàn toàn "sạch", loại bỏ rủi ro bị crash (sụp đổ) khi đẩy vào các thuật toán khắt khe như XGBoost.

## 4. BIẾN ĐỔI LOGARIT (LOGARITHMIC TRANSFORMATION)
Các biến `speechiness` và `instrumentalness` có cấu trúc phân phối bị lệch phải. Việc trực tiếp sử dụng dữ liệu này sẽ làm giảm khả năng khái quát hóa của mô hình tuyến tính và mô hình cây. 
Áp dụng phép biến đổi logarit tự nhiên: $X'_{log} = \log(X + 1)$. Việc cộng 1 giúp ngăn ngừa lỗi toán học $\log(0) = -\infty$.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for idx, col in enumerate(['speechiness', 'instrumentalness']):
    # Phân phối gốc
    sns.histplot(df[col], bins=40, ax=axes[idx, 0], color='salmon', kde=True)
    axes[idx, 0].set_title(f'Phân phối gốc: {col}', fontsize=12)
    axes[idx, 0].set_xlabel('Giá trị')
    axes[idx, 0].set_ylabel('Tần suất')
    
    # Biến đổi Log1p
    df[f'{col}_log'] = np.log1p(df[col])
    
    # Phân phối sau biến đổi
    sns.histplot(df[f'{col}_log'], bins=40, ax=axes[idx, 1], color='mediumseagreen', kde=True)
    axes[idx, 1].set_title(f'Phân phối sau Log1p: {col}_log', fontsize=12)
    axes[idx, 1].set_xlabel('Giá trị sau logarit')
    axes[idx, 1].set_ylabel('Tần suất')

plt.tight_layout()
plt.show()

# Xóa các cột chưa biến đổi để giải phóng không gian đặc trưng
df = df.drop(columns=['speechiness', 'instrumentalness'])


### Giải thích, Nhận xét & Đánh giá Chuyên sâu: Biến đổi Phân phối Logarit

1. GIẢI THÍCH:
Hệ thống áp dụng phép biến đổi toán học Logarit tự nhiên cộng 1: $X'_{log} = \log(X + 1)$ lên hai biến có độ lệch phải cao là `speechiness` và `instrumentalness`. Lớp Đồ họa (Seaborn) sau đó so sánh trực quan hàm Mật độ Xác suất (PDF) trước và sau khi biến đổi.

2. NHẬN XÉT:
Hình ảnh phân phối phản ánh sự cải thiện rõ rệt. Phép biến đổi Logarit đã giãn nở các giá trị tập trung dày đặc ở cận 0 và nén các giá trị ngoại lai ở đuôi phân phối (Long Tail) lại gần nhau. Thay vì một cái đuôi kéo dài dị dạng, dữ liệu sau khi biến đổi đã sở hữu một đường cong trơn tru hơn (Smoother Curve), tiệm cận gần hơn với đặc tính của phân phối chuẩn (Gaussian-like Distribution).

3. ĐÁNH GIÁ (HIGH IMPACT):
Sự chuyển đổi hình thái này giải tỏa một điểm nghẽn nghiêm trọng cho mô hình Hồi quy Tuyến tính (Linear Regression) vốn phụ thuộc vào giả định phần dư chuẩn (Normality of Residuals). Đối với các mô hình cây quyết định (Tree-based Models), biến đổi logarit cũng giúp thuật toán phân nhánh (Splitting Criteria) hoạt động hiệu quả hơn, từ đó tăng độ chính xác của dự báo tổng thể.

## 5. CHUẨN HÓA ĐẶC TRƯNG (MIN-MAX SCALING)
Chuẩn hóa dữ liệu là thủ tục cần thiết để loại bỏ ảnh hưởng của thang đo và biên độ giá trị. Thuật toán `MinMaxScaler` thực hiện phép chuyển đổi tuyến tính sau:
$$X_{scaled} = \frac{X - X_{min}}{X_{max} - X_{min}}$$
Phương pháp này bảo toàn quan hệ hình học tuyến tính giữa các điểm dữ liệu và đảm bảo mọi thuộc tính đầu vào đều có miền giá trị $[0, 1]$.

In [ ]:
TARGET = 'target_popularity'
FEATURES = [c for c in df.columns if c not in [TARGET, 'track_id']]

scaler = MinMaxScaler()
df[FEATURES] = scaler.fit_transform(df[FEATURES])

print("Tóm tắt thống kê các đặc trưng sau khi Chuẩn hóa (Min-Max):")
display(df[FEATURES].describe().loc[['min', 'max', 'mean', 'std']].round(4).T)


### Giải thích, Nhận xét & Đánh giá Chuyên sâu: Đồng bộ hóa Thang đo (Min-Max Scaling)

1. GIẢI THÍCH:
Toàn bộ các đặc trưng định lượng (Quantitative Features) được đưa qua bộ chuyển đổi tuyến tính `MinMaxScaler` của Scikit-Learn. Thuật toán này ép biên độ của tất cả các cột về cùng một không gian hình học đa chiều, trong đó giá trị nhỏ nhất của mỗi cột là 0 và giá trị lớn nhất là 1.

2. NHẬN XÉT:
Bảng kết quả thống kê mô tả sau khi chuẩn hóa xác nhận sự thành công của thuật toán: Dòng `min` đồng loạt hiển thị 0.0000 và dòng `max` đồng loạt hiển thị 1.0000. Dữ liệu đã rũ bỏ hoàn toàn rào cản về Đơn vị đo lường (Units of Measurement). Một bài hát dài `2000` phút hay `0.5` phút giờ đây cũng mang trọng số ảnh hưởng ngang bằng với độ to (Loudness) là `-60` dB.

3. ĐÁNH GIÁ (CRITICAL IMPACT):
Đây là công đoạn "Làm phẳng Không gian" (Space Flattening) sống còn. Nếu không có nó, các thuật toán sử dụng Gradient Descent (như Neural Networks hoặc XGBoost nâng cao) sẽ mất hàng ngày để hội tụ vì quỹ đạo tối ưu (Optimization Trajectory) sẽ dao động dích dắc. Việc chuẩn hóa này đảm bảo không có đặc trưng nào có thể "bắt nạt" (Dominate) các đặc trưng khác chỉ vì nó mang những con số to hơn.

## 6. KẾT LUẬN
Giai đoạn Feature Engineering đã hoàn thành với tập dữ liệu đạt tiêu chuẩn:
- Loại bỏ hoàn toàn giá trị Null.
- Các phân phối phi tuyến đã được điều chỉnh.
- Toàn bộ vector đặc trưng đã được đồng bộ thang đo.

Dữ liệu hiện tại đã sẵn sàng để chuyển giao sang giai đoạn Huấn luyện Mô hình (Notebook 06).